# Sink removal across 10 VQA examples (SmolVLM2) — does it generalise?

Runs the full pipeline (rater -> visual, all layers) on **10 (image, question)**
examples, builds a **global position-bias baseline** (the question-invariant
sink), subtracts it (B) + drops the top sink patches (A), and reports
**quantitative** evidence that it generalises:

* how often the raw importance peaks on the same sink patch,
* how often debiasing moves the peak off the sink,
* and — the key metric — whether same-image maps become **question-specific**
  (cosine similarity of raw vs debiased maps for the same image, different
  questions: raw ~1 = all sink; debiased lower = conditioned).

> **Runtime:** GPU runtime. SmolVLM2 is open (no token needed).

## 1. Install + clone

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words pytest matplotlib

In [ ]:
!rm -rf text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git
%cd text_vision_attention_map

## 2. Setup + the 10 VQA examples
Four images (cats, bus, two-people, dog); several with two questions so we can
test question-conditioning on the SAME image.

In [ ]:
import importlib.util, os, math
import numpy as np
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS
from transformers import AutoProcessor
tokenizer = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-2.2B-Instruct").tokenizer

CATS = "http://images.cocodataset.org/val2017/000000039769.jpg"
BUS  = "https://ultralytics.com/images/bus.jpg"
ZID  = "https://ultralytics.com/images/zidane.jpg"
DOG  = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"

EXAMPLES = [
    (CATS, "How many cats are in the image?"),
    (CATS, "Where is the remote control?"),
    (BUS,  "What color is the bus?"),
    (BUS,  "How many people are in the image?"),
    (BUS,  "Where is the bus?"),
    (ZID,  "How many people are in the image?"),
    (ZID,  "What is the man wearing?"),
    (DOG,  "What animal is in the image?"),
    (DOG,  "Where is the dog?"),
    (DOG,  "What color is the dog?"),
]
DROP_K = 3          # A: hard-drop the top-3 sink patches

def to_grid(vec, L_v):
    g = int(round(math.sqrt(L_v)))
    if g * g != L_v:
        g = math.ceil(math.sqrt(L_v)); vec = np.concatenate([vec, np.zeros(g*g-L_v, vec.dtype)])
    return vec.reshape(g, g)

def up(a, size, mode=Image.BILINEAR):
    a = (a / (a.max() + 1e-9) * 255).astype('uint8')
    return np.array(Image.fromarray(a).resize(size, mode))

def importance_for(url, question, _cache={}):
    img = _cache.get(url) or S.load_image(url); _cache[url] = img
    o = S.make_smolvlm_output(image=img, question=question)
    if o is None:
        return None
    maps, tpos, _ = RS.sliced_maps_from_full(o.raw_scores, o.image_token_mask, o.text_token_mask)
    tt = tokenizer.convert_ids_to_tokens(o.input_ids[tpos].tolist())
    rr = RS.select_important_text_tokens(maps, text_tokens=tt, tokenizer=tokenizer,
                                         question=question, pct=0.5)
    imp, _, _, _ = VS.image_importance(maps, rr.rater_mask)   # score, no threshold
    return dict(url=url, img=img, question=question, raters=rr.kept_tokens(tt), importance=imp)

print("examples:", len(EXAMPLES))

## 3. Run all 10, build the global sink baseline, and debias
First run downloads the ~4.5 GB weights.

In [ ]:
data = [importance_for(u, q) for (u, q) in EXAMPLES]
data = [d for d in data if d is not None]
assert data, "no examples produced a result"
L_v = data[0]["importance"].numel()
assert all(d["importance"].numel() == L_v for d in data), "L_v differs across examples"

# GLOBAL baseline: mean importance over ALL examples -> the question/image-invariant sink
baseline = VS.make_baseline([d["importance"] for d in data])
sink = int(baseline.argmax())
sink_mask = VS.sink_token_mask(baseline, DROP_K)

for d in data:
    deb = VS.subtract_baseline(d["importance"], baseline)          # B
    deb = deb.clone(); deb[sink_mask] = 0.0                        # A
    s = deb.sum();  deb = deb / s if s > 0 else deb
    d["debiased"] = deb

conc = (baseline.max() / baseline.mean()).item()
print(f"global sink patch index : {sink}  (of {L_v})   concentration max/mean = {conc:.1f}x")
print(f"drop_sink_k = {DROP_K}  ->  sink patches: {sink_mask.nonzero().squeeze(-1).tolist()}")

## 4. Quantitative summary

In [ ]:
n = len(data)
raw_on_sink = sum(int(d["importance"].argmax()) == sink for d in data)
moved = sum(int(d["debiased"].argmax()) != sink for d in data)
uniq_raw = len({int(d["importance"].argmax()) for d in data})
uniq_deb = len({int(d["debiased"].argmax()) for d in data})

print(f"raw peak == global sink        : {raw_on_sink}/{n}   (sink dominates the raw maps)")
print(f"debiased peak moved off sink   : {moved}/{n}")
print(f"distinct peaks  raw -> debiased: {uniq_raw} -> {uniq_deb}   (higher = more content-specific)")
print()
print(f"{'question':<34}{'raters':<24}{'raw':>4}{'deb':>5}")
print('-' * 68)
for d in data:
    print(f"{d['question'][:33]:<34}{str(d['raters'])[:23]:<24}"
          f"{int(d['importance'].argmax()):>4}{int(d['debiased'].argmax()):>5}")

## 5. Key metric: does debiasing make same-image maps question-specific?
For each image with >1 question, cosine similarity of the two maps. Raw maps are
near-identical (all sink -> ~1.0); debiased maps should be **less** similar.

In [ ]:
def cos(a, b):
    return F.cosine_similarity(a.view(1, -1), b.view(1, -1)).item()

groups = defaultdict(list)
for d in data:
    groups[d["url"]].append(d)

raw_sims, deb_sims = [], []
for url, ds in groups.items():
    for i in range(len(ds)):
        for j in range(i + 1, len(ds)):
            raw_sims.append(cos(ds[i]["importance"], ds[j]["importance"]))
            deb_sims.append(cos(ds[i]["debiased"], ds[j]["debiased"]))

print(f"same-image, different-question map similarity  ({len(raw_sims)} pairs):")
print(f"   raw      cosine : {np.mean(raw_sims):.3f}   (high  -> NOT question-specific)")
print(f"   debiased cosine : {np.mean(deb_sims):.3f}   (lower -> question-specific)")
print(f"   drop             : {np.mean(raw_sims) - np.mean(deb_sims):+.3f}")

## 6. Visual grid: image | raw importance | debiased (B + A)

In [ ]:
fig, axes = plt.subplots(len(data), 3, figsize=(11, 3.4 * len(data)))
if len(data) == 1:
    axes = axes[None, :]
for r, d in enumerate(data):
    img = d["img"]
    raw_h = to_grid(d["importance"].numpy(), L_v)
    deb_h = to_grid(d["debiased"].numpy(), L_v)
    axes[r, 0].imshow(img); axes[r, 0].axis("off")
    axes[r, 0].set_title(f"Q: {d['question']}\nraters: {d['raters']}", fontsize=8)
    axes[r, 1].imshow(img); axes[r, 1].imshow(up(raw_h, img.size), cmap="jet", alpha=0.5)
    axes[r, 1].set_title("raw importance (sink)", fontsize=9); axes[r, 1].axis("off")
    axes[r, 2].imshow(img); axes[r, 2].imshow(up(deb_h, img.size), cmap="jet", alpha=0.5)
    axes[r, 2].set_title("debiased (B + A)", fontsize=9); axes[r, 2].axis("off")
plt.tight_layout(); plt.show()